# CosMx UMAP embedding and Leiden clustering

**Goal:** Embed the merged, QC-filtered CosMx AnnData into UMAP space and compute
Leiden clusters. Tonsil control cores are separated prior to embedding so the UMAP
reflects tumor tissue biology only. The resulting object is the entry point for
cell typing in `cosmx_cell_typing`.

**Input:** `outputs/processed/combined_adata_qc_filtered.h5ad` — merged QC-filtered
CosMx AnnData from `filtering_and_merging_metadata`.

**Output:** `outputs/processed/cosmx_combined_umap.h5ad` — CosMx tumor cells with
UMAP coordinates and Leiden cluster labels.

## Setup

In [ ]:
# standard libraries
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell and spatial analysis
import anndata
import scanpy as sc
import squidpy as sq
import decoupler as dc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42  # embed fonts in PDF — required by most journals
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes — consistent across all analyses
# MHC II pos = orange (#FF8811), MHC II neg = purple (#462255)
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # neg first
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']  # pos first
cmap_umap     = ['#9DD9D2FF', '#462255FF', '#FF8811FF', '#046E8FFF', '#D44D5CFF']  # used in CosMx/scRNA UMAPs

In [ ]:
# load paths from central config
# collaborators: update this path to point to your local copy of paths_config.yml
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
cosmx_filtered_path = Path(cfg['datasets']['cosmx']['processed']) / 'combined_adata_qc_filtered.h5ad'
salcher_ref_path    = Path(cfg['datasets']['salcher_atlas']['cosmx_subset'])

# outputs
processed_out = repo_root / cfg['outputs']['processed']
fig_dir       = repo_root / cfg['outputs']['figures']
table_dir     = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir / 'figure3'
supp_out  = fig_dir / 'figure3-supp'
table_out = table_dir / 'figure3'

processed_out.mkdir(parents=True, exist_ok=True)
fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

sc.settings.figdir = supp_out

## Load data

The QC-filtered CosMx AnnData is loaded and genes are subset to those present
in the Salcher atlas reference panel (975 of the original 1000 CosMx genes).
This ensures the gene space is consistent with the reference used for cell typing.

In [ ]:
# reload reference after rename fix
adata_ref = anndata.read_h5ad(salcher_ref_path)
print(f"Reference var_names sample: {adata_ref.var_names[:5].tolist()}")
print(f"BBLN in ref: {'BBLN' in adata_ref.var_names}")
print(f"RGS5 in ref: {'RGS5' in adata_ref.var_names}")

# now subset
adata = anndata.read_h5ad(cosmx_filtered_path)
adata = adata[:, adata.var_names.isin(adata_ref.var_names)].copy()
print(f"Genes after matching: {adata.n_vars}")

In [ ]:
adata

## Remove tonsil control cores

Tonsil cores were included on the TMA as staining controls. They are separated
from the tumor tissue before UMAP embedding so the embedding reflects LUAD
biology only. The tonsil subset is retained as a separate object for QC reference.

In [ ]:
# separate tonsil control cores from tumor tissue
tonsil_adata = adata[adata.obs['region'] == 'tonsil'].copy()
adata = adata[adata.obs['region'] != 'tonsil'].copy()

print(f"Tumor cells: {adata.n_obs:,}")
print(f"Tonsil control cells: {tonsil_adata.n_obs:,}")

## Normalization and preprocessing

Raw counts are preserved in `adata.layers['counts']` before normalization.
The preprocessing steps mirror those in `filtering_and_merging_metadata` but
are applied here to the gene-matched, tonsil-excluded subset in preparation
for UMAP embedding.

In [ ]:
# restore raw counts before normalization
adata.X = adata.layers['counts'].copy()

# normalize, log-transform, PCA, neighbors
adata.layers['counts'] = adata.X.copy()  # keep raw counts for DESeq2
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)

In [ ]:
import scipy.sparse as sp
import numpy as np

X = adata.layers['counts']
if sp.issparse(X):
    X = X.toarray()

print(f"Min: {X.min():.3f}")
print(f"Max: {X.max():.3f}")
print(f"Sample values: {X[0, :10]}")

# These are not log-transformed, this was a false positive warning from scanpy 

## UMAP and Leiden clustering

UMAP is computed with `min_dist=0.3` and `spread=1.5` to produce a layout that
separates cell populations clearly while maintaining local structure. Leiden
clustering is run at default resolution as a starting point for cell type assignment.
The result is saved to `outputs/processed/cosmx_umap.h5ad` and used as a checkpoint
— subsequent runs load the pre-computed embedding rather than recomputing.
This file is the primary input for `cosmx_cell_typing`.

In [ ]:
%%time

umap_path = processed_out / 'cosmx_umap.h5ad'

if umap_path.exists():
    # load pre-computed UMAP instead of recomputing
    print("Loading pre-computed UMAP...")
    adata = anndata.read_h5ad(umap_path)
    print(f"Loaded: {adata.n_obs:,} cells, {adata.obs['leiden'].nunique()} Leiden clusters")
else:
    # compute UMAP and Leiden clustering — takes ~20-30 min on 700k cells
    print("Computing UMAP and Leiden clustering...")
    sc.tl.umap(adata, min_dist=0.3, spread=1.5)
    sc.tl.leiden(adata)
    print(f"Leiden clusters: {adata.obs['leiden'].nunique()}")
    
    # save checkpoint
    adata.write(umap_path)
    print(f"Saved → {umap_path}")

## UMAP and Leiden clustering

UMAP is computed with `min_dist=0.3` and `spread=1.5` to produce a layout that
separates cell populations clearly while maintaining local structure. Leiden
clustering is run at default resolution as a starting point for cell type assignment.
The result is saved to `outputs/processed/cosmx_umap.h5ad` and used as a checkpoint
— subsequent runs load the pre-computed embedding rather than recomputing.
This file is the primary input for `cosmx_cell_typing`.

Note: `vmin=0` is set on the CD68/CK8-18 protein channel plot as one cell has a
negative intensity value (likely a segmentation artifact) which otherwise compresses
the color scale.

In [ ]:
# inspection plots — sanity check before saving
sc.settings.figdir = supp_out

sc.pl.umap(adata, color='leiden', palette='tab20', s = 1.5,
           title='Leiden Clusters', save='_umap_leiden.pdf')

sc.pl.umap(adata, color='Mean.PanCK', cmap='viridis', vmax=60000, s = 1.5,
           title='Mean PanCK Protein', save='_umap_panck.pdf')

sc.pl.umap(adata, color='Mean.CD45', cmap='viridis', vmax=8000, s = 1.5,
           title='Mean CD45 Protein', save='_umap_cd45.pdf')

sc.pl.umap(adata, color='Mean.CD68_CK8_18', cmap='viridis', vmin = 0, vmax=20000, s = 1.5,
           title='Mean CD68 Protein', save='_umap_cd68.pdf')